# Notebook 04: Feature Engineering
##### Football Match Prediction 
##### Student Name: Vishal Chaudhary
##### Student Number: X23332794

### IMPORT REQUIRED LIBRARIES

In [1]:
import pandas as pd
import numpy as np
import gc
from datetime import datetime, timedelta
from pathlib import Path
import warnings
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("Libraries imported successfully!")
print(f"Processing Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Libraries imported successfully!
Processing Date: 2025-11-14 01:01:33



### LOAD DATASETS

In [2]:
# Define paths
DATA_RAW_PATH = Path('/Users/vishalchaudhary/Desktop/Football_Project_Final/DOMAIN_APPLICATIONS_PROJECT/data/raw')
PROCESSED_PATH = Path('/Users/vishalchaudhary/Desktop/Football_Project_Final/DOMAIN_APPLICATIONS_PROJECT/data/processed')
FEATURES_PATH = Path('/Users/vishalchaudhary/Desktop/Football_Project_Final/DOMAIN_APPLICATIONS_PROJECT/data/features')
RESULTS_PATH = Path('/Users/vishalchaudhary/Desktop/Football_Project_Final/DOMAIN_APPLICATIONS_PROJECT/results')
FIGURES_PATH = RESULTS_PATH / 'figures'


FEATURES_PATH.mkdir(parents=True, exist_ok=True)

df_merged = pd.read_csv(FEATURES_PATH/'df_merged_cleaned.csv')
df_fixture_stats_clean = pd.read_csv('/Users/vishalchaudhary/Desktop/Football_Project/DOMAIN_APPLICATIONS_PROJECT/data/processed/df_fixture_stats_clean.csv')
df_top_players_raw = pd.read_csv('/Users/vishalchaudhary/Desktop/Football_Project/DOMAIN_APPLICATIONS_PROJECT/data/raw/top_500_player_stats_2018_2023.csv')



### Handle Outliers

In [3]:
# Cap injury_count at 95th percentile
injury_cap = df_merged['injury_count'].quantile(0.95)
df_merged['injury_count'] = df_merged['injury_count'].clip(upper=injury_cap)
df_merged['away_injury_count'] = df_merged['away_injury_count'].clip(upper=injury_cap)

# Stabilize points_per_game by increasing denominator
df_merged['home_points_per_game'] = df_merged['total_points'] / (df_merged['total_wins'] + df_merged['h2h_draws'] + df_merged['h2h_away_wins'] + 10)
df_merged['away_points_per_game'] = df_merged['away_total_points'] / (df_merged['total_wins'] + df_merged['h2h_draws'] + df_merged['h2h_away_wins'] + 10)

# --- Step 3: Additional H2H Features ---

# H2H last 5 win percentage
df_merged['h2h_last_5_home_win_pct'] = df_merged['h2h_last_5_home_wins'] / 5
df_merged['h2h_last_5_draw_pct'] = df_merged['h2h_last_5_draws'] / 5
df_merged['h2h_last_5_away_win_pct'] = df_merged['h2h_last_5_away_wins'] / 5

# H2H goal difference
df_merged['h2h_goal_diff'] = df_merged['h2h_avg_home_goals'] - df_merged['h2h_avg_away_goals']


### Team Form Features (Rolling Averages)

In [4]:
# Sort by date and team for rolling calculations
df_merged = df_merged.sort_values(['home_team_id', 'date'])

# Rolling average for home team (last 5 games)
df_merged['home_avg_goals_last_5'] = df_merged.groupby('home_team_id')['home_goals'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)
df_merged['home_avg_shots_last_5'] = df_merged.groupby('home_team_id')['home_shots_on_goal'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)
df_merged['home_avg_points_last_5'] = df_merged.groupby('home_team_id')['total_points'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)

# Sort by away_team_id for away team rolling calculations
df_merged = df_merged.sort_values(['away_team_id', 'date'])
df_merged['away_avg_goals_last_5'] = df_merged.groupby('away_team_id')['away_goals'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)
df_merged['away_avg_shots_last_5'] = df_merged.groupby('away_team_id')['away_shots_on_goal'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)
df_merged['away_avg_points_last_5'] = df_merged.groupby('away_team_id')['away_total_points'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)


 ### Squad Stability Features

In [5]:
# Injury ratio
df_merged['home_injury_ratio'] = df_merged['injury_count'] / df_merged['squad_size']
df_merged['away_injury_ratio'] = df_merged['away_injury_count'] / df_merged['away_squad_size']

# Team experience proxy
df_merged['home_team_experience'] = df_merged['avg_squad_age'] * df_merged['total_team_minutes']
df_merged['away_team_experience'] = df_merged['away_avg_squad_age'] * df_merged['away_total_team_minutes']


### Tactical Features

In [6]:
# Simplified formation encoding (extract number of defenders)
df_merged['home_formation_defenders'] = df_merged['home_formation'].str.split('-').str[0].astype(float)
df_merged['away_formation_defenders'] = df_merged['away_formation'].str.split('-').str[0].astype(float)
df_merged['home_formation_defenders'] = df_merged['home_formation_defenders'].fillna(df_merged['home_formation_defenders'].mode()[0])
df_merged['away_formation_defenders'] = df_merged['away_formation_defenders'].fillna(df_merged['away_formation_defenders'].mode()[0])


### Prepare for Modeling

In [7]:
# Define feature columns
numeric_features = [
    'home_shots_on_goal', 'away_shots_on_goal', 'home_total_shots', 'away_total_shots',
    'home_corners', 'away_corners', 'avg_team_rating', 'away_avg_team_rating',
    'injury_count', 'away_injury_count', 'squad_size', 'away_squad_size',
    'avg_squad_age', 'away_avg_squad_age', 'total_points', 'away_total_points',
    'h2h_home_win_pct', 'h2h_avg_total_goals', 'h2h_rating_diff', 'h2h_points_diff',
    'h2h_injury_diff', 'home_rating_rank_ratio', 'away_rating_rank_ratio',
    'home_points_per_game', 'away_points_per_game', 'h2h_last_5_home_win_pct',
    'h2h_goal_diff', 'home_avg_goals_last_5', 'home_avg_shots_last_5',
    'home_avg_points_last_5', 'away_avg_goals_last_5', 'away_avg_shots_last_5',
    'away_avg_points_last_5', 'home_injury_ratio', 'away_injury_ratio',
    'home_team_experience', 'away_team_experience', 'home_formation_defenders',
    'away_formation_defenders'
]
categorical_features = ['league_name', 'home_formation', 'away_formation']

# Handle any remaining missing values
df_merged[numeric_features] = df_merged[numeric_features].fillna(df_merged[numeric_features].median())
df_merged[categorical_features] = df_merged[categorical_features].fillna(df_merged[categorical_features].mode().iloc[0])

# Create preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Apply preprocessing
X = df_merged[numeric_features + categorical_features]
y = df_merged['target']
X_preprocessed = preprocessor.fit_transform(X)

### Comprehensive Quality Check

In [8]:
def data_quality_check(df, name):
    print(f"\n Quality Check for {name} \n")
    print(f"Shape: {df.shape}")
    print(f"\nData Types:\n{df.dtypes}\n")
    print(f"\nMissing Values per Column:\n{df.isna().sum()}\n")
    total_missing = df.isna().sum().sum()
    print(f"Total Missing Values: {total_missing} ({total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}% of data)\n")
    print(f"\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\nUnique Values in Key Columns:")
    key_cols = ['fixture_id', 'date', 'league_id', 'home_team_id', 'away_team_id']
    for col in key_cols:
        if col in df.columns:
            print(f"{col}: {df[col].nunique()} unique values")
    print(f"\nDescriptive Statistics for Numeric Columns:\n{df[numeric_features].describe()}\n")

data_quality_check(df_merged, "df_merged")


 Quality Check for df_merged 

Shape: (18894, 160)

Data Types:
fixture_id                    int64
date                         object
timestamp                     int64
league_id                     int64
league_name                  object
                             ...   
away_injury_ratio           float64
home_team_experience        float64
away_team_experience        float64
home_formation_defenders    float64
away_formation_defenders    float64
Length: 160, dtype: object


Missing Values per Column:
fixture_id                  0
date                        0
timestamp                   0
league_id                   0
league_name                 0
                           ..
away_injury_ratio           0
home_team_experience        0
away_team_experience        0
home_formation_defenders    0
away_formation_defenders    0
Length: 160, dtype: int64

Total Missing Values: 22506 (0.74% of data)


Number of Duplicates: 0


Unique Values in Key Columns:
fixture_id: 18894 unique

### Save Preprocessed Data

In [9]:
df_merged.to_csv(FEATURES_PATH/'df_merged_engineered.csv', index=False)
np.save(FEATURES_PATH/'X_preprocessed.npy', X_preprocessed)
np.save(FEATURES_PATH/'y.npy', y)
print("Saved engineered dataset to 'df_merged_engineered.csv' and preprocessed features to 'X_preprocessed.npy', 'y.npy'")


Saved engineered dataset to 'df_merged_engineered.csv' and preprocessed features to 'X_preprocessed.npy', 'y.npy'


In [10]:
# Ensure consistent dtypes
df_merged['date'] = pd.to_datetime(df_merged['date'])
df_fixture_stats_clean['date'] = pd.to_datetime(df_fixture_stats_clean['date'])
df_merged['season'] = df_merged['season'].astype(str)
df_top_players_raw['season'] = df_top_players_raw['season'].astype(str)

### create missing features

In [11]:
# create missing features: games played, wins, draws, losses
home_games = df_merged.groupby(['home_team_id', 'season']).agg({
    'fixture_id': 'count', 'target': lambda x: (x == 2).sum()
}).reset_index()
home_games.columns = ['home_team_id', 'season', 'home_games_played', 'home_wins']

home_draws = df_merged[df_merged['target'] == 1].groupby(['home_team_id', 'season'])['fixture_id'].count().reset_index()
home_draws.columns = ['home_team_id', 'season', 'home_draws']
home_games = home_games.merge(home_draws, on=['home_team_id', 'season'], how='left')
home_games['home_draws'] = home_games['home_draws'].fillna(0)
home_games['home_loses'] = home_games['home_games_played'] - home_games['home_wins'] - home_games['home_draws']

away_games = df_merged.groupby(['away_team_id', 'season']).agg({
    'fixture_id': 'count', 'target': lambda x: (x == 0).sum()
}).reset_index()
away_games.columns = ['away_team_id', 'season', 'away_games_played', 'away_wins']

away_draws = df_merged[df_merged['target'] == 1].groupby(['away_team_id', 'season'])['fixture_id'].count().reset_index()
away_draws.columns = ['away_team_id', 'season', 'away_draws']
away_games = away_games.merge(away_draws, on=['away_team_id', 'season'], how='left')
away_games['away_draws'] = away_games['away_draws'].fillna(0)
away_games['away_loses'] = away_games['away_games_played'] - away_games['away_wins'] - away_games['away_draws']

df_merged = df_merged.merge(home_games, on=['home_team_id', 'season'], how='left')
df_merged = df_merged.merge(away_games, on=['away_team_id', 'season'], how='left')

### Per-game metrics

In [12]:
df_merged['home_goals_per_game'] = df_merged['total_goals_for'] / df_merged['home_games_played'].replace(0, 1)
df_merged['away_goals_per_game'] = df_merged['away_total_goals_for'] / df_merged['away_games_played'].replace(0, 1)
df_merged['home_goals_conceded_per_game'] = df_merged['total_goals_against'] / df_merged['home_games_played'].replace(0, 1)
df_merged['away_goals_conceded_per_game'] = df_merged['away_total_goals_against'] / df_merged['away_games_played'].replace(0, 1)

### Goal difference metrics

In [13]:
df_merged['goal_diff_home'] = df_merged['total_goals_for'] - df_merged['total_goals_against']
df_merged['goal_diff_away'] = df_merged['away_total_goals_for'] - df_merged['away_total_goals_against']
df_merged['goal_diff_difference'] = df_merged['goal_diff_home'] - df_merged['goal_diff_away']


### Rolling goals conceded (last 5 games)

In [14]:
df_merged = df_merged.sort_values(['home_team_id', 'date'])
df_merged['home_goals_conceded_L5'] = df_merged.groupby('home_team_id')['away_goals'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)
df_merged = df_merged.sort_values(['away_team_id', 'date'])
df_merged['away_goals_conceded_L5'] = df_merged.groupby('away_team_id')['home_goals'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)

### Form difference

In [15]:
df_merged['form_difference'] = df_merged['home_avg_points_last_5'] - df_merged['away_avg_points_last_5']

### Rolling stats from fixture stats

In [16]:
df_fixture_stats_sorted = df_fixture_stats_clean.sort_values(['home_team_id', 'date']).copy()
rolling_stats_home = []

for team_id in df_fixture_stats_sorted['home_team_id'].unique():
    team_home = df_fixture_stats_sorted[df_fixture_stats_sorted['home_team_id'] == team_id].copy()
    team_away = df_fixture_stats_sorted[df_fixture_stats_sorted['away_team_id'] == team_id].copy()

    team_home_stats = team_home[['fixture_id', 'date', 'home_shots_on_goal', 'home_total_shots',
                                 'home_corners', 'home_possession_numeric', 'home_passes_accurate',
                                 'home_passes_percent_numeric']].copy()
    team_home_stats.columns = ['fixture_id', 'date', 'shots_on_goal', 'total_shots', 'corners',
                               'possession', 'passes_accurate', 'passes_percent']

    team_away_stats = team_away[['fixture_id', 'date', 'away_shots_on_goal', 'away_total_shots',
                                 'away_corners', 'away_possession_numeric', 'away_passes_accurate',
                                 'away_passes_percent_numeric']].copy()
    team_away_stats.columns = ['fixture_id', 'date', 'shots_on_goal', 'total_shots', 'corners',
                               'possession', 'passes_accurate', 'passes_percent']

    team_all = pd.concat([team_home_stats, team_away_stats]).sort_values('date')
    team_all['team_id'] = team_id

    team_all['rolling_shots_on_goal'] = team_all['shots_on_goal'].rolling(window=5, min_periods=1).mean().shift(1)
    team_all['rolling_total_shots'] = team_all['total_shots'].rolling(window=5, min_periods=1).mean().shift(1)
    team_all['rolling_corners'] = team_all['corners'].rolling(window=5, min_periods=1).mean().shift(1)
    team_all['rolling_possession'] = team_all['possession'].rolling(window=5, min_periods=1).mean().shift(1)
    team_all['rolling_passes_accurate'] = team_all['passes_accurate'].rolling(window=5, min_periods=1).mean().shift(1)
    team_all['rolling_pass_accuracy'] = team_all['passes_percent'].rolling(window=5, min_periods=1).mean().shift(1)

    rolling_stats_home.append(team_all[['fixture_id', 'team_id', 'rolling_shots_on_goal',
                                        'rolling_total_shots', 'rolling_corners',
                                        'rolling_possession', 'rolling_passes_accurate',
                                        'rolling_pass_accuracy']])

df_rolling_stats = pd.concat(rolling_stats_home, ignore_index=True)

rolling_home = df_rolling_stats.copy()
rolling_home.columns = ['fixture_id', 'home_team_id', 'home_rolling_shots_on_goal',
                        'home_rolling_total_shots', 'home_rolling_corners',
                        'home_rolling_possession', 'home_rolling_passes_accurate',
                        'home_rolling_pass_accuracy']
rolling_away = df_rolling_stats.copy()
rolling_away.columns = ['fixture_id', 'away_team_id', 'away_rolling_shots_on_goal',
                        'away_rolling_total_shots', 'away_rolling_corners',
                        'away_rolling_possession', 'away_rolling_passes_accurate',
                        'away_rolling_pass_accuracy']

df_merged = df_merged.merge(rolling_home, on=['fixture_id', 'home_team_id'], how='left')
df_merged = df_merged.merge(rolling_away, on=['fixture_id', 'away_team_id'], how='left')


### Difference features for rolling stats

In [17]:
df_merged['shots_on_goal_diff'] = df_merged['home_rolling_shots_on_goal'] - df_merged['away_rolling_shots_on_goal']
df_merged['total_shots_diff'] = df_merged['home_rolling_total_shots'] - df_merged['away_rolling_total_shots']
df_merged['corners_diff'] = df_merged['home_rolling_corners'] - df_merged['away_rolling_corners']
df_merged['possession_diff'] = df_merged['home_rolling_possession'] - df_merged['away_rolling_possession']
df_merged['pass_accuracy_diff'] = df_merged['home_rolling_pass_accuracy'] - df_merged['away_rolling_pass_accuracy']


### Shot efficiency

In [18]:
df_merged['home_shot_efficiency'] = df_merged['total_goals_for'] / df_merged['home_total_shots'].replace(0, 1)
df_merged['away_shot_efficiency'] = df_merged['away_total_goals_for'] / df_merged['away_total_shots'].replace(0, 1)
df_merged['home_shot_efficiency'] = df_merged['home_shot_efficiency'].clip(upper=10)
df_merged['away_shot_efficiency'] = df_merged['away_shot_efficiency'].clip(upper=10)
df_merged['shot_efficiency_diff'] = df_merged['home_shot_efficiency'] - df_merged['away_shot_efficiency']


### Days since last game

In [19]:
df_merged = df_merged.sort_values(['home_team_id', 'date'])
df_merged['home_days_since_last'] = df_merged.groupby('home_team_id')['date'].diff().dt.days.fillna(7)
df_merged = df_merged.sort_values(['away_team_id', 'date'])
df_merged['away_days_since_last'] = df_merged.groupby('away_team_id')['date'].diff().dt.days.fillna(7)
df_merged['days_rest_difference'] = df_merged['home_days_since_last'] - df_merged['away_days_since_last']


### Table and match metrics

In [20]:
df_merged['table_proximity'] = abs(df_merged['avg_rank'] - df_merged['away_avg_rank'])
df_merged['is_close_match'] = (df_merged['table_proximity'] <= 3).astype(int)


### League-level metrics

In [21]:
league_home_wins = df_merged[df_merged['target'] == 2].groupby('league_id').size() / df_merged.groupby('league_id').size()
df_merged['league_home_advantage'] = df_merged['league_id'].map(league_home_wins).fillna(league_home_wins.median())
league_avg_goals = df_merged.groupby('league_id')['total_goals'].mean()
df_merged['league_avg_goals'] = df_merged['league_id'].map(league_avg_goals).fillna(league_avg_goals.median())
league_points_std = df_merged.groupby('league_id')['total_points'].std()
df_merged['competitive_balance'] = df_merged['league_id'].map(league_points_std).fillna(league_points_std.median())

### Top players

In [22]:
top_players = df_top_players_raw[df_top_players_raw['rating'] > 6.5].groupby(['team', 'season']).size().reset_index(name='num_top_players')
df_merged = df_merged.merge(
    top_players.rename(columns={'num_top_players': 'home_num_top_players', 'team': 'home_team_name'}),
    on=['home_team_name', 'season'], how='left'
)
df_merged = df_merged.merge(
    top_players.rename(columns={'num_top_players': 'away_num_top_players', 'team': 'away_team_name'}),
    on=['away_team_name', 'season'], how='left'
)
df_merged['home_num_top_players'] = df_merged['home_num_top_players'].fillna(0)
df_merged['away_num_top_players'] = df_merged['away_num_top_players'].fillna(0)


### Balance and composite metrics

In [23]:
df_merged['rank_balance'] = df_merged['table_proximity']
df_merged['form_balance'] = df_merged['form_difference']
df_merged['overall_balance'] = (
    abs(df_merged['h2h_rating_diff']) + abs(df_merged['h2h_points_diff']) + abs(df_merged['form_difference'])
) / 3
df_merged['both_defensive'] = (
    (df_merged['total_goals_against'] < df_merged['total_goals_against'].quantile(0.25)) &
    (df_merged['away_total_goals_against'] < df_merged['away_total_goals_against'].quantile(0.25))
).astype(int)
df_merged['defensive_sum'] = df_merged['total_goals_against'] + df_merged['away_total_goals_against']
df_merged['combined_attack'] = df_merged['total_goals_for'] + df_merged['away_total_goals_for']
df_merged['low_scoring_likelihood'] = (
    df_merged['h2h_avg_total_goals'] < df_merged['h2h_avg_total_goals'].quantile(0.25)
).astype(int)
df_merged['home_draw_tendency'] = df_merged['h2h_draw_pct']
df_merged['away_draw_tendency'] = df_merged['h2h_draw_pct']
df_merged['combined_draw_tendency'] = df_merged['h2h_draw_pct']
df_merged['dominance_score'] = (
    0.4 * df_merged['avg_team_rating'] +
    0.3 * df_merged['total_points'] / df_merged['total_points'].max() * 100 +
    0.3 * df_merged['total_goals_for'] / df_merged['total_goals_for'].max() * 100
)


### Handle missing values

In [24]:
new_features = [
    'home_games_played', 'away_games_played', 'home_draws', 'away_draws', 'home_loses', 'away_loses',
    'home_goals_per_game', 'away_goals_per_game', 'home_goals_conceded_per_game', 'away_goals_conceded_per_game',
    'goal_diff_home', 'goal_diff_away', 'goal_diff_difference', 'home_goals_conceded_L5', 'away_goals_conceded_L5',
    'home_num_top_players', 'away_num_top_players',
    'home_rolling_shots_on_goal', 'away_rolling_shots_on_goal', 'shots_on_goal_diff',
    'home_rolling_total_shots', 'away_rolling_total_shots', 'total_shots_diff',
    'home_rolling_corners', 'away_rolling_corners', 'corners_diff',
    'home_rolling_possession', 'away_rolling_possession', 'possession_diff',
    'home_rolling_passes_accurate', 'away_rolling_passes_accurate',
    'home_rolling_pass_accuracy', 'away_rolling_pass_accuracy', 'pass_accuracy_diff',
    'home_shot_efficiency', 'away_shot_efficiency', 'shot_efficiency_diff',
    'home_days_since_last', 'away_days_since_last', 'days_rest_difference',
    'table_proximity', 'is_close_match',
    'league_home_advantage', 'league_avg_goals', 'competitive_balance',
    'rank_balance', 'form_balance', 'overall_balance',
    'both_defensive', 'defensive_sum', 'combined_attack',
    'low_scoring_likelihood', 'home_draw_tendency', 'away_draw_tendency',
    'combined_draw_tendency', 'dominance_score'
]
df_merged[new_features] = df_merged[new_features].fillna(df_merged[new_features].median())


### Validate shot efficiency

In [25]:
print("\nShot Efficiency Validation:")
print(f"Max home_shot_efficiency: {df_merged['home_shot_efficiency'].max():.2f}")
print(f"Max away_shot_efficiency: {df_merged['away_shot_efficiency'].max():.2f}")
print(f"Rows with home_shot_efficiency > 5: {(df_merged['home_shot_efficiency'] > 5).sum()}")
print(f"Rows with away_shot_efficiency > 5: {(df_merged['away_shot_efficiency'] > 5).sum()}")



Shot Efficiency Validation:
Max home_shot_efficiency: 10.00
Max away_shot_efficiency: 10.00
Rows with home_shot_efficiency > 5: 5449
Rows with away_shot_efficiency > 5: 7890


### Data quality check

In [26]:
def data_quality_check(df, name, new_features, feature_columns):
    print(f"\n Quality Check for {name} \n")
    print(f"Shape: {df.shape}")
    print(f"\nData Types for New Features:\n{df[new_features].dtypes}\n")
    print(f"\nMissing Values per New Feature:\n{df[new_features].isna().sum()}\n")
    total_missing = df[new_features].isna().sum().sum()
    print(f"Total Missing Values in New Features: {total_missing} ({total_missing / (df.shape[0] * len(new_features)) * 100:.2f}%)\n")
    print(f"\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\nUnique Values in Key Columns:")
    key_cols = ['fixture_id', 'date', 'league_id', 'home_team_id', 'away_team_id']
    for col in key_cols:
        if col in df.columns:
            print(f"{col}: {df[col].nunique()} unique values")
    print(f"\nDescriptive Statistics for New Features:\n")
    print(df[new_features].describe().to_string())
    if feature_columns:
        missing_features = [col for col in feature_columns if col not in df.columns]
        print(f"\nMissing Feature Columns from feature_columns list: {missing_features}")


### Save final dataset

In [27]:
feature_columns = new_features.copy()
data_quality_check(df_merged, 'df_merged_complete_v2', new_features, feature_columns)

df_merged.to_csv(FEATURES_PATH/'df_merged_complete_v2.csv', index=False)
gc.collect()


 Quality Check for df_merged_complete_v2 

Shape: (18894, 219)

Data Types for New Features:
home_games_played                 int64
away_games_played                 int64
home_draws                      float64
away_draws                      float64
home_loses                      float64
away_loses                      float64
home_goals_per_game             float64
away_goals_per_game             float64
home_goals_conceded_per_game    float64
away_goals_conceded_per_game    float64
goal_diff_home                    int64
goal_diff_away                    int64
goal_diff_difference              int64
home_goals_conceded_L5          float64
away_goals_conceded_L5          float64
home_num_top_players            float64
away_num_top_players            float64
home_rolling_shots_on_goal      float64
away_rolling_shots_on_goal      float64
shots_on_goal_diff              float64
home_rolling_total_shots        float64
away_rolling_total_shots        float64
total_shots_diff          

0